In [1]:
import os  # Used for file management (removing files, path operations)
import pandas as pd  # Used for handling tabular data (reading/writing CSV files)
import subprocess  # Used to run PHANOTATE as an external process
from tqdm import tqdm  # Used to display a progress bar for tracking processing
from os import listdir  # Used to list files in a directory
from Bio import SeqIO  # Used to read and parse FASTA files
from Bio.Seq import Seq  # Used for reverse complementing sequences

In [2]:
dir_path = '~/projects/Yuzhen/PB_interactions/'
phage_file = 'data/phage_genomes/A1a.fasta'
phanotate_path = '~/.local/bin/phanotate.py'
phage_path = os.path.join(dir_path, phage_file)

In [3]:
### takes about 6 seconds for first file
def run_phanotate(phage_path, phanotate_path, output_file):
    """
    Processes a single phage genome using PHANOTATE and saves the gene predictions to a CSV.

    INPUTS:
    - input_fasta (str): Path to the single FASTA file containing the phage genome.
    - phanotate_path (str): Path to the PHANOTATE executable/script.
    - output_csv (str): Path to the output CSV file where gene predictions will be saved.

    OUTPUT:
    - CSV file with columns ['phage_ID', 'gene_ID', 'gene_sequence'].
    """

    # extract phage name from file (remove directory and .fasta)
    phage_name = os.path.basename(phage_path).replace('.fasta', '')
    
    # run phanotate on input fasta file

    raw_str = f"{phanotate_path} {phage_path}"
    process = subprocess.Popen(raw_str, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    stdout, stderr = process.communicate()

    # process phanotate output
    std_splits = stdout.split(sep=b'\n')[2:]
    return std_splits
    



In [9]:
std_splits = run_phanotate(phage_path, phanotate_path, 'test')

In [ ]:
temp_tsv_path = 'temp_phanotate_output.tsv'
with open(temp_tsv_path, 'w') as temp_tab:
    for split in std_splits:
        split = split.replace(b',',b'') # remove commas for pandas compatability
        temp_tab.write(split.decode('utf-8') + '\n')

orfs = pd.read_csv(temp_tsv_path, sep='\t', lineterminator='\n', index_col=False)
orfs.head()

,#START,STOP,FRAME,CONTIG,SCORE
0,1,552,+,A1a,-5.967169e+02
1,562,2943,+,A1a,-4.095286e+09
2,2946,3548,+,A1a,-4.242529e+02
3,3564,6248,+,A1a,-1.094128e+12
4,6298,9996,+,A1a,-1.780538e+16


In [12]:
print(orfs.iloc[0])

#START           1
STOP           552
FRAME            +
CONTIG         A1a
SCORE    -596.7169
Name: 0, dtype: object


In [6]:
sequence = str(SeqIO.read(phage_file, 'fasta').seq)

name_list = []
gene_list = []
gene_ids = []
count = 1

In [7]:
for j, strand in enumerate(orfs['FRAME']):
    start = orfs['#START'][j]
    stop = orfs['STOP'][j]
    
    if strand == '+':
        gene = sequence[start-1:stop]
    else:
        sequence_part = sequence[stop-1:start]
        gene = str(Seq(sequence_part).reverse_complement())

    phage_name = os.path.basename(phage_path).replace('.fasta', '')

    name_list.append(phage_name)
    gene_list.append(gene)
    gene_ids.append(f"{phage_name}_gp{count}")
    count += 1



In [13]:
for i in range(len(gene_ids)):
    if i < 2:
        print(name_list[i])
        print(gene_ids[i])
        print(gene_list[i])
        print()

A1a
A1a_gp1
TTAGACGCTGTGAACCTGACGTTAGAAGCCCTGGGGGAGTCTCGCGTTATGGATATCAACACTTCAAACCCAAGCGCAGGGTTAGCACGTTCTGCACTCGCGCGTAATCGCCGAGGCCTGCTAAGCACTGGCTACTGGTTCAACGTAGTCGAGCGAGAGGTTACTCCTACGACTGACGGACTTATTAAGGTTCCGTGGAACCAGTTGGCTGTGTATGATGCGTGCTCCGACAATAAGTACGGTGTACGCAATGGGAACCTTTACGACCTGGTAGAGCAGAACGAGTACTTCGACTCACCTGTTAAAATTAAGGTAGTGCTGGACCTCAACTTTGAGGACCTGCCGGAGCACGCGGCTATGTGGATTGCAAACTACACCACTGCGCAGGTGTACCTGAACGACCTCGGCAGTGACGGCAACTACGCCAATTACGCCTCTGAGGCGGAGCGATACAAGGCCCTGGTGCTGCGCGAGCATCTGCGTAACCAGAAGTACAGCACCAGCAAGACCAGATTCGCACGTCGTATCCGTCGTGCACGCTTCATGATTTAA

A1a
A1a_gp2
ATGGCGCAATCATTAGAAGGCACCATTCAGAGTCTGCTCCAGGGCGTGTCCCAGCAGATTCCAAGAGAGCGCCAGCCTGGGCAACTTGGGGCGCAGCTGAACATGCTCAGTGACCCAGTATCCGGATTACGTAGACGGCCCCCAGCAGAGATTGTGTGGGAGAGCAGCATCGACAATCCGGGCCTCGACTCCCTGTTCACAGAATACGTCGAGCGCGGCACTGACGGTAGGCACCTGCTGATTAACACCAGCAACGGAAATTGGTGGCTCCTGTCCAAGAACGGCAAGACCATAGTAAACAGTGGGAACGACCCTTACTTCGTAACCACAGTAGGGCAGACCTCCATTCAGACTGCGAGTATCGCCGGGCTGACCTACATCCTGAATACGGAGATGGCCCCGAGTACGACTGTAGATAACAC

In [14]:
genebase = pd.DataFrame(list(zip(name_list, gene_ids, gene_list)), columns=['phage_ID', 'gene_ID', 'gene_sequence'])
genebase.to_csv('data/phanotate/genebase.csv', index=False)